This notebook demonstrates basic operations with Milvus DB

### Setup requirement prerequisites

In [ ]:
!pip install --no-deps --ignore-installed -r ../requirements.txt -v

### 03.01. Connecting to Milvus

In [ ]:
#Creating a connection

#Import the pymilvus package
from pymilvus import connections

#Create list of connections
connections.add_connection(
    #Specify a name for the connection
    learn={    
        "host": "localhost",
        "port": "19530",
        "username" : "",
        "password" : ""
    })

#Connect
connection_id="learn"
connections.connect(connection_id)

#List all connections
connections.list_connections()

### 03.02. Creating databases and users

In [ ]:
#Database operations
from pymilvus import db

#Get current list of databases available to the connection
current_dbs=db.list_database(using=connection_id)
print("Current databases: ", current_dbs)

db_name="course_db"

if ( db_name not in current_dbs):
    print("Creating database :", db_name)
    wiki_db = db.create_database(db_name, using=connection_id) 
    
#Switch to use the new database
db.using_database(db_name, using=connection_id)


In [ ]:
#user management
from pymilvus import Role,utility

current_users=utility.list_usernames(using=connection_id)
print("Current user list: ", current_users)

new_user = "course_public"

if new_user not in current_users:
    utility.create_user(new_user, "password", using=connection_id)

#Assign a role to the user
public_role = Role("public", using=connection_id)
print(" Role public exists? ", public_role.is_exist())

#Add user to role
public_role.add_user(new_user)


### 03.03. Creating collections

In [ ]:
from pymilvus import CollectionSchema, FieldSchema, DataType, Collection
import json

#Define fields
course_id = FieldSchema(
    name="course_id",
    dtype=DataType.INT64,
    is_primary=True,
    max_length=32)

title= FieldSchema(
    name="title",
    dtype=DataType.VARCHAR,
    max_length=256)

description= FieldSchema(
    name="description",
    dtype=DataType.VARCHAR,
    max_length=2048)

#Dim should match the embedding size 
desc_embedding = FieldSchema(
    name="desc_embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=1536
)

#Define schema
wiki_schema=CollectionSchema(
    fields=[course_id, title, description, desc_embedding],
    description="Courses List",
    enable_dynamic_field=True
)

collection_name="courses_list"

#Creation collection
wiki_collection=Collection(
    name=collection_name,
    schema=wiki_schema,
    using=connection_id,
    shard_num=2
)

from pymilvus import utility

#List all collections
print("Current collections: ",utility.list_collections(using=connection_id))

#setup existing collection into another object
r_collection=Collection(collection_name, using=connection_id)
print("\n", r_collection.schema)


### 03.04. Inserting data into Milvus

In [ ]:
#read the input course CSV
import pandas as pd
course_descriptions = pd.read_csv("course-descriptions.csv")
course_descriptions.head() # Display first few rows of course_description, default number of rows = 5

In [ ]:
#Use langchain to create embeddings.
from langchain_openai import OpenAIEmbeddings
import os
from config import OPENAI_API_KEY

#model="text-embedding-3-small": Balanced accuracy and cost, 1536 dimensions.
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
#Prepare data for insert

i_course_id = course_descriptions["Course ID"].tolist()
i_title = course_descriptions["Title"].tolist()
i_description = course_descriptions["Description"].tolist()

i_desc_embedding=[embeddings_model.embed_query(i) for i in i_description]

#Format for data input
insert_data=[i_course_id, i_title, i_description, i_desc_embedding]

In [ ]:
#Initiate a collection object and insert data
course_collection = Collection(collection_name,using=connection_id)

#Insert
mr=course_collection.insert(insert_data)

#Flush the data after insert
print("Inserted data. Now flushing")
course_collection.flush(timeout=180)


### 03.05. Build an index

In [ ]:
#Build an index
index_params = {
    "metric_type":"L2",
    "index_type":"IVF_FLAT",
    "params" :{"nlist":1024}
}

course_collection.create_index(
    field_name="desc_embedding",
    index_params=index_params
)

utility.index_building_progress(collection_name,using=connection_id)



### 03.06. Querying scalar data

In [ ]:
# To view in Attu
# Select Collections on left menu
# Select Click on "unloaded" next to courses_list and select Load.
# Click on courses_list > Open 'Data Preview' Tab

#Load the Collection
# NOTE: A collection should first be loaded into memory before
#       queries can be executed against it

course_collection.load()
print("Course collection loaded..")

In [ ]:
q_result= course_collection.query(
    expr = "course_id == 1001",
    output_fields = ["title","description"]
)
print(q_result)
print("\n Result object :", type(q_result[0]))

In [ ]:
q_result2= course_collection.query(
    expr = "(title LIKE 'MLOps%') && (course_id > 1001) ",
    output_fields = ["title","description"]
)
print(q_result2)

### 03.07. Searching Vector fields

In [ ]:
#Make sure that the collection is already loaded.

#  nprobe controls how many clusters Milvus searches. Higher nprobe = more accurate but slower. Lower nprobe = faster but less accurate
search_params = {
    "metric_type": "L2", 
    "offset": 0, 
    "ignore_growing": False, 
    "params": {"nprobe": 10}
}

#Embed the input search string
search_string = "machine learning"
search_embed=embeddings_model.embed_query(search_string)

#Perform search
s_results=course_collection.search(
    data=[search_embed], #input query to search for
    anns_field="desc_embedding", #field to search with ANN
    param=search_params,
    limit=10, #Limit output
    expr=None, #Use additional scalar conditions
    output_fields=["title"],
    consistency_level="Strong"
)

print("Search result object:", type(s_results[0]),"\n")
#Print results in order of match
for i in s_results[0]:
    print(i.id, str(round(i.distance, 2)), "\t",i.entity.get("title"))

In [ ]:
#Search an unrelated query

#Embed the input search string
search_string2 = "best movies of the year"
search_embed2=embeddings_model.embed_query(search_string2)

#Perform search
s_results2=course_collection.search(
    data=[search_embed2], #input query to search for
    anns_field="desc_embedding", #field to search with ANN
    param=search_params,
    limit=10, #Limit output
    expr=None, #Use additional scalar conditions
    output_fields=["title"],
    consistency_level="Strong"
)

#Print results in order of match
for i in s_results2[0]:
    print(i.id, str(round(i.distance, 2)), "\t",i.entity.get("title"))

### 03.08. Deleting objects and entities

In [ ]:
#Delete a single record
course_collection.delete("course_id in [1002]")

In [ ]:
#Drop a collection
utility.drop_collection(collection_name,using=connection_id)

In [ ]:
#drop a database
#Make sure to drop all collections in the database first

db.drop_database(db_name, using=connection_id)